# 配体/多肽：从晶体结构分离 → 参数化 → 得到可直接并入 MD 的 itp

配套：`MD_pipeline.ipynb`（拿到 itp 之后建体系）、`../references/variants.md`

三种情形，处理方式完全不同：

| 情形 | 例子 | 做法 |
|---|---|---|
| **A. 小分子配体**（非共价） | BTN、ATP | 单独提取 → GAFF2/AM1-BCC → 作为独立 moleculetype |
| **B. 共价修饰的肽/蛋白片段** | 生物素化赖氨酸肽（本项目 PEPB） | **把被修饰残基的侧链一起切进配体**，形成"共轭物"整体参数化；蛋白侧改用不含该侧链的残基 |
| **C. 非共价肽** | 底物肽 | 按普通肽处理，用 pdb2gmx，不必 ACPYPE |

**B 是本 notebook 的重点**：直接对 BTN 单独参数化再试图用键连到 Lys 上，是错的（电荷、缺氢、连接处的键/角/二面角全会缺失）。

In [ ]:
import os, re, subprocess, shutil, json

# ======================= CONFIG =======================
CRYSTAL_PDB = "/root/autodl-tmp/GROMACS/WT/input/WT.pdb"   # 原始晶体/复合物
WORKDIR     = "/root/autodl-tmp/GROMACS/LIGPAR"            # 工作目录
LIG_NAME    = "PEPB"          # 配体残基名 / ACPYPE 的 -b 名
MODE        = "B"             # A=小分子配体  B=共价修饰肽(共轭物)  C=非共价肽
# --- MODE A: 小分子 ---
HET_CHAIN   = "D"             # 配体所在链（或 resname）
HET_RESNAME = "BTN"
NET_CHARGE  = 0               # 配体净电荷（必须自己确认！）
# --- MODE B: 共轭物（肽 + 修饰基团）---
PEPTIDE_CHAIN = "C"           # 肽所在链
PEPTIDE_RESI  = (10, 14)      # 肽的残基编号范围
MODIFIED_RESI = 12            # 被共价修饰的残基（如 Lys12）
# ======================================================

ACPY  = "/root/autodl-tmp/mmpbsa/bin/acpype"      # 与 ambertools 同环境
OBABEL = shutil.which("obabel") or "/usr/bin/obabel"
os.makedirs(WORKDIR, exist_ok=True)
print("WORKDIR:", WORKDIR, "| MODE:", MODE, "| acpype:", os.path.exists(ACPY))

## 1 提取

按链/残基名挑出原子，保留原始坐标与 `CONECT`（键连信息是后续定键级的唯一依据）。

In [ ]:
def parse_pdb(path):
    atoms, conect, order = {}, {}, []
    for ln in open(path, errors="ignore"):
        rec = ln[:6].strip()
        if rec in ("ATOM", "HETATM"):
            try:
                aid = int(ln[6:11])
            except ValueError:
                continue
            atoms[aid] = ln.rstrip("\n")
            order.append(aid)
        elif rec == "CONECT":
            nums = [int(x) for x in ln[6:].split()]
            if nums:
                conect.setdefault(nums[0], set()).update(nums[1:])
    return atoms, conect, order

atoms, conect, order = parse_pdb(CRYSTAL_PDB)
print(f"读入 {len(atoms)} 个原子, {len(conect)} 条 CONECT")

def chain_of(ln):  return ln[21]
def resname_of(ln): return ln[17:20].strip()
def resi_of(ln):   return int(ln[22:26])

if MODE == "A":
    sel = [a for a in order if chain_of(atoms[a]) == HET_CHAIN and resname_of(atoms[a]) == HET_RESNAME]
elif MODE == "B":
    lo, hi = PEPTIDE_RESI
    sel = [a for a in order
           if chain_of(atoms[a]) == PEPTIDE_CHAIN and lo <= resi_of(atoms[a]) <= hi]
    # 共轭基团若在别的链（如 BTN 在 D 链），一并纳入
    sel += [a for a in order if chain_of(atoms[a]) == HET_CHAIN and resname_of(atoms[a]) == HET_RESNAME]
else:
    raise SystemExit("MODE C（非共价肽）请直接用 pdb2gmx，不需要 ACPYPE")

sel = sorted(set(sel))
print("选中原子:", len(sel), "| 残基:", sorted({resname_of(atoms[a]) for a in sel}))
with open(f"{WORKDIR}/lig_raw.pdb", "w") as f:
    for i, a in enumerate(sel, 1):
        f.write(atoms[a][:6] + f"{i:5d}" + atoms[a][11:] + "\n")
print("→ lig_raw.pdb")

In [ ]:
# ---- 关键检查 1：配体是否与蛋白有共价键 ----
sel_set = set(sel)
covalent = [(a, b) for a in sel for b in conect.get(a, ())
            if b in atoms and b not in sel_set and b in atoms]
print("跨界共价键 (配体原子 ↔ 蛋白原子):", covalent[:10], "..." if len(covalent) > 10 else "")

# ---- 关键检查 2：元素组成与重原子数 ----
from collections import Counter
def elem_of(ln):
    e = ln[76:78].strip()
    return e if e else re.sub(r"[^A-Za-z]", "", ln[12:16])[:1].upper()
cnt = Counter(elem_of(atoms[a]) for a in sel)
print("元素组成:", dict(cnt), "| 重原子数:", sum(v for k, v in cnt.items() if k != "H"))
if covalent:
    print("\n⚠ 检测到共价连接：这是 MODE B（共轭物）。")
    print("   正确做法：把被修饰残基的侧链一起切进配体（本 cell 已按整条肽提取），")
    print("   蛋白侧在建体系时改用不含该侧链的残基（或自定义 rtp），不要把 BTN 单独参数化后硬连。")

## 2 定键级（最容易出错的一步）

ACPYPE **不会**自己猜对键级：它依赖 mol2 里的键类型。

- 有 `CONECT` 且结构简单 → `obabel` 转换通常可用
- 复杂/芳香/带电 → **给 SMILES，再用 RDKit 在晶体坐标上赋键级**（本 notebook 走这条）
- 拿到 mol2 后必须人工核对：芳香环、羰基、酰胺、电荷形式

> 本项目实测：晶体里 BTN 的脲羰基 `C–O` 被精修成 1.41 Å（正常 1.23 Å），若直接信坐标会在参数化时得到错误的键级/电荷。**以化学 SMILES 为准，坐标只用来定位原子。**

In [ ]:
# MODE A/B 通用：用 SMILES + 晶体坐标生成带正确键级的 mol2
# 需要 rdkit：micromamba install -c conda-forge rdkit
SMILES = None      # ← 填上！例如生物素酰基肽的 SMILES
# 本项目实际用的 SMILES（LGKGG + 生物素酰胺，N端NH3+/C端COO-）：
SMILES = ("[NH3+][C@@H](CC(C)C)C(=O)NCC(=O)N[C@@H](CCCCNC(=O)CCCC[C@@H]1SC[C@@H]2[C@H]1NC(=O)N2)"
          "C(=O)NCC(=O)NCC(=O)[O-]")

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem
    m = Chem.MolFromSmiles(SMILES) if SMILES else None
    print("SMILES 解析:", "OK" if m else "失败", "| 原子数:", m.GetNumAtoms() if m else 0,
          "| 形式电荷:", Chem.GetFormalCharge(m) if m else None)
    if m:
        print("提示：先用 RDKit 按 SMILES 生成 3D，再与晶体坐标做原子映射；")
        print("      或手工确认 lig_raw.pdb 的原子顺序与 SMILES 一致后直接写 mol2。")
except ImportError:
    print("rdkit 未安装：micromamba install -c conda-forge rdkit")

In [ ]:
# 简化路径：用 obabel 从 PDB(含 CONECT) 生成 mol2，再人工核对键级
# （本项目最终采用的是手工整理好的 conj.mol2：晶体坐标 + 正确键级）
rc = subprocess.run([OBABEL, f"{WORKDIR}/lig_raw.pdb", "-O", f"{WORKDIR}/lig_obabel.mol2"],
                    capture_output=True, text=True)
print("obabel rc:", rc.returncode, rc.stderr[-300:])
print(open(f"{WORKDIR}/lig_obabel.mol2").read()[:600] if os.path.exists(f"{WORKDIR}/lig_obabel.mol2") else "no output")

## 3 ACPYPE 参数化（GAFF2 + AM1-BCC）

```bash
acpype -i conj.mol2 -c bcc -a gaff2 -n 0 -b PEPB -o gmx
```

- `-c bcc`：AM1-BCC 电荷（用 sqm 半经验计算，几十秒到几分钟）
- `-a gaff2`：与 Amber 力场（amber99sb-ildn）配套
- `-n`：**净电荷，必须与 SMILES 的形式电荷一致**，否则 sqm 不收敛或电荷错误
- `-b`：输出前缀，同时决定 itp 里的 `[moleculetype]` 名

In [ ]:
SRC_MOL2 = f"{WORKDIR}/conj.mol2"      # ← 放你核对好的 mol2（晶体坐标 + 正确键级）
if not os.path.exists(SRC_MOL2):
    print(f"缺 {SRC_MOL2}：先把 lig_raw.pdb 整理成带正确键级的 mol2")

VERBOSE = True
if os.path.exists(SRC_MOL2):
    cmd = [ACPY, "-i", os.path.basename(SRC_MOL2), "-c", "bcc", "-a", "gaff2",
           "-n", str(NET_CHARGE), "-b", LIG_NAME, "-o", "gmx", "-v", "2" if VERBOSE else "0"]
    r = subprocess.run(cmd, cwd=WORKDIR, capture_output=True, text=True)
    print("acpype rc:", r.returncode)
    print((r.stdout + r.stderr)[-1500:])

## 4 校验 itp（必做）

四个数对得上才能用：**原子数**、**电荷和 = 净电荷**、**residue 名**、**atomtypes 不与力场冲突**。

In [ ]:
import numpy as np
def check_itp(itp, expect_charge=0, tag=""):
    atoms, sec, q = [], None, []
    for ln in open(itp):
        s = ln.strip()
        if s.startswith("["):
            sec = s.strip("[] \t"); continue
        if sec == "atoms" and s and not s.startswith(";"):
            p = s.split()
            if len(p) >= 6 and p[0].isdigit():
                atoms.append((int(p[0]), p[1], p[4]))
                q.append(float(p[6]))
    print(f"[{tag}] 原子 {len(atoms)} | 电荷和 {sum(q):+.4f} (期望 {expect_charge:+d}) | "
          f"resnames {sorted({a[2] for a in atoms})}")
    return len(atoms), sum(q)

ITP = f"{WORKDIR}/{LIG_NAME}.acpype/{LIG_NAME}_GMX.itp"
if os.path.exists(ITP):
    n, sq = check_itp(ITP, NET_CHARGE, LIG_NAME)
    assert abs(sq - NET_CHARGE) < 0.01, "电荷和与净电荷不符：检查 -n 与 SMILES 形式电荷"
    print("✓ itp 校验通过")
else:
    print("itp 尚未生成（acpype 未跑或失败）")

## 5 并入 MD 体系

```python
INC = f'#include "{LIG_NAME}.acpype/{LIG_NAME}_GMX.itp"'
# 放在力场 include 之后、第一个 [moleculetype] 之前（否则 [atomtypes] 顺序非法）
# [molecules] 里追加一行：  {LIG_NAME}  1
```

坐标合并：`prot.gro`（pdb2gmx 产物）+ `PEPB_GMX.gro`（ACPYPE 产物），按顺序拼接并重编号，
注意 **ACPYPE 的 gro 用的是它自己的坐标系**，若来自晶体坐标则与蛋白同框，可直接拼；否则需先对齐。

详见 `MD_pipeline.ipynb` 的 M3b。